In [3]:
#Exercise 1

import json
import re
import pandas as pd
 
INPUT_FILE = "house data/House.html"
OUTPUT_FILE = "bengaluru_houses.csv"
 
 
html = open(INPUT_FILE, encoding="utf-8").read()
print("File read. Size:", len(html), "characters")
 

marker = "window.__initialData__="
start = html.find(marker) + len(marker)
depth = 0
for end in range(start, len(html)):
    if html[end] == "{":
        depth += 1
    elif html[end] == "}":
        depth -= 1
        if depth == 0:
            break
data = json.loads(html[start:end + 1])
print("JSON loaded successfully")
 
 
properties = data["srp"]["pageData"]["properties"]
print("Number of listings found:", len(properties))
 
 
def price_in_lakh(rupees):
    """21998600 -> 220.0 lakh"""
    try:
        return round(float(rupees) / 100000, 2)
    except (TypeError, ValueError):
        return None
 
 
def to_number(value):
    """'2600' -> 2600.0 ; None -> None"""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None
 
 
rows = []
for p in properties:
    # Some listings are a price RANGE (e.g. "2.25 - 3.28 Cr").
    # We keep the minimum price, and also record the text version.
    rows.append({
        "location":       p.get("LOCALITY"),
        "bhk":            to_number(p.get("BEDROOM_NUM")),
        "sqft":           to_number(p.get("LOCALIZED_AREA_VALUE")),
        "area_unit":      p.get("LOCALIZED_AREA_UNIT_LABEL"),
        "price_lakh":     price_in_lakh(p.get("MIN_PRICE")),
        "price_text":     p.get("PRICE"),
        "price_per_sqft": to_number(p.get("PRICE_SQFT")),
        "property_name":  p.get("PROP_NAME"),
    })
 
houses = pd.DataFrame(rows)
 
print("\nAll areas in sqft?", (houses["area_unit"] == "sqft").all())
print("Missing values:\n", houses.isna().sum().to_string())
 
houses.to_csv(OUTPUT_FILE, index=False)
print("\nSaved", len(houses), "rows to", OUTPUT_FILE)
print(houses.head(10).to_string(index=False))


File read. Size: 1774012 characters
JSON loaded successfully
Number of listings found: 27

All areas in sqft? True
Missing values:
 location          0
bhk               0
sqft              0
area_unit         0
price_lakh        0
price_text        0
price_per_sqft    0
property_name     0

Saved 27 rows to bengaluru_houses.csv
                        location  bhk   sqft area_unit  price_lakh      price_text  price_per_sqft              property_name
          Bommasandra, Bangalore  5.0 2600.0      sqft      219.99          2.2 Cr          8461.0                SKR Gardens
5th Block Hbr Layout, HBR Layout 12.0 1200.0      sqft      380.00          3.8 Cr         31666.0                           
        Sarjapur Road, Bangalore  4.0 3040.0      sqft      225.00 2.25  - 3.28 Cr          9087.0 CasaLife by Bhavisha Homes
           Whitefield, Bangalore  5.0 4629.0      sqft      381.38 3.81  - 6.65 Cr         11299.0                 DSR Elixir
           Whitefield, Bangalore  4.0 4

In [5]:
#Exercise 2

# 1 # 2
import requests
import pandas as pd
import time

URL = "https://api.open-meteo.com/v1/forecast"
params2 = {
    "latitude": 12.2958 ,
    "longitude": 76.6394,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "timezone": "Asia/Kolkata",
    "forecast_days": 7,
}

try:
    r = requests.get(URL, params=params2, timeout=20)
    result = r.json()
    wx = pd.DataFrame(result["hourly"])       # JSON -> DataFrame
    wx["time"] = pd.to_datetime(wx["time"])
    print("Collected", len(wx), "hourly records")
    
except Exception as e:
    wx = pd.DataFrame()
    print("No internet — the next cell will create sample data.")


wx.to_csv("weather_mysore.csv", index=False)
wx.head()

# 3.it shows abcd is not defined
#4. 3 cities exercise


cities = {
    "Bengaluru": (12.9716, 77.5946),
    "Mysuru": (12.2958, 76.6394),
    "Chennai": (13.0827, 80.2707)
}

all_weather = []

for city, (latitude, longitude) in cities.items():

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": "Asia/Kolkata",
        "forecast_days": 7
    }

    try:
        r = requests.get(URL, params=params, timeout=20)
        result = r.json()

        wx = pd.DataFrame(result["hourly"])

        # Convert time from string to datetime
        wx["time"] = pd.to_datetime(wx["time"])

        # Add city name
        wx["city"] = city

        # Store this city's DataFrame
        all_weather.append(wx)

        print(city, ":", len(wx), "records")

    except Exception as e:
        print("Error for", city, ":", e)

    # Wait before making the next API request
    time.sleep(1)

# Combine all 3 DataFrames
weather_df = pd.concat(all_weather, ignore_index=True)


print("Total records:", len(weather_df))
weather_df.head()

Collected 168 hourly records
Bengaluru : 168 records
Mysuru : 168 records
Chennai : 168 records
Total records: 504


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,city
0,2026-08-20 00:00:00,21.5,92,0.2,10.1,Bengaluru
1,2026-08-20 01:00:00,21.6,93,0.2,10.3,Bengaluru
2,2026-08-20 02:00:00,21.4,93,0.0,11.0,Bengaluru
3,2026-08-20 03:00:00,21.2,94,0.0,10.4,Bengaluru
4,2026-08-20 04:00:00,21.0,94,0.0,9.5,Bengaluru
